In [16]:
import sys
!{sys.executable} -m pip install requests pyspark

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached pyspark-4.1.1.tar.gz (455.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ---------------------------------------- 0.0/41.7 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.7 kB ? eta -:--:--
     ------------------ ------------------- 20.5/41.7 kB 162.5 kB/s eta 0:00:01
     -------------------------------------  41.0/41.7 kB 245.8 kB/s eta 0:00:01
     -------------------------------------- 41.7/41.7 kB 223.4 kB/s eta 0:00:00
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.wh

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import sys
from pyspark.sql import SparkSession

# --- 1. CONFIGURAÇÃO PARA WINDOWS (IMPORTANTE) ---
# Se você instalou o winutils em c:/hadoop/bin:
os.environ['HADOOP_HOME'] = "C:/hadoop"
sys.path.append("C:/hadoop/bin")

# --- 2. INICIALIZAÇÃO DO SPARK LOCAL ---
try:
    spark = SparkSession.builder \
        .appName("Teste Local VS Code") \
        .config("spark.driver.host", "localhost") \
        .getOrCreate()
    print("✅ Spark inicializado com sucesso no VS Code!")
except Exception as e:
    print(f"❌ Erro crítico: Você ainda não configurou o Java ou Hadoop. Detalhes: {e}")
    spark = None

# --- 3. FUNÇÃO DE EXTRAÇÃO ---
def extract_clientes_local():
    if spark is None:
        return
    
    # IMPORTANTE: Mude este caminho para um arquivo que esteja no seu HD!
    # O VS Code não lê "/Volumes/..." da nuvem diretamente.
    caminho_local = "C:/usuarios/seu_usuario/downloads/clientes_amostra.parquet"
    
    if not os.path.exists(caminho_local):
        print(f"⚠️ Arquivo não encontrado em: {caminho_local}")
        print("Dica: Baixe o arquivo do Databricks e coloque o caminho correto aqui.")
        return

    try:
        df = spark.read.load(caminho_local)
        df.show(10)
        return df
    except Exception as e:
        print(f"Erro ao ler arquivo local: {e}")

# Execução
extract_clientes_local()

Aviso: Spark não disponível localmente. Erro: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

[ERRO] O Spark não foi inicializado. Não é possível ler o Volume localmente.

Processamento finalizado. Data: 2026-04-02


In [10]:
!pip install requests pyspark


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Users\Arilson\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [5]:
dbutils.widgets.text("data_execucao", "")
data_execucao = dbutils.widgets.get("data_execucao")

NameError: name 'dbutils' is not defined

In [6]:
import requests
from pyspark.sql.functions import lit

ModuleNotFoundError: No module named 'requests'

In [0]:
def extract_data(date, base="BRL"):

  url = f"https://api.apilayer.com/exchangerates_data/{date}&base={base}"

  headers = {
  "apikey": "<api-token>"
 }

  parametros = {"base":base}
 
  response = requests.request("GET", url, headers=headers, params = parametros)

  if response.status_code != 200:
    raise Exception(f"Não foi possivel extrair os dados!!! | status_code {response.status_code}")

  return response.json()

In [0]:
def data_to_df (data_json): 
    
    data_tuple = [(moeda, float (taxa)) for moeda, taxa in data_json["rates"].items()]
    return data_tuple

In [0]:
def save_to_parquet(converted_data):

  ano, mes, dia = converted_data['date'].split('-')
  full_path = f"dbfs:/databricks-results/bronze/{ano}/{mes}/{dia}/"

  response = data_to_df(converted_data)
  df = spark.createDataFrame(response, ["moeda", "taxa"])
  df = df.withColumn('data', lit(f"{ano}-{mes}-{dia}"))

  df.write.mode("overwrite").format("parquet").save(full_path)

  print(f"Os arquivos foram salvos em {full_path}")

In [0]:
cotas = extract_data(data_execucao)
save_to_parquet(cotas)